In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔄 API Versioning & Backwards Compatibility Guide

**Managing API evolution while maintaining backwards compatibility and supporting multiple client versions**

This guide covers:
- API versioning strategies (URL, header, query param)
- Semantic versioning
- Deprecation policies
- Breaking changes management
- Client migration paths
- Backwards compatibility testing
- API gateway versioning
- Version lifecycle management
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. API Versioning Strategies

### Versioning Approaches
```python
from dataclasses import dataclass
from typing import Dict, List, Optional, Callable
from enum import Enum
from datetime import datetime, timedelta

class VersioningStrategy(Enum):
    """API versioning approaches"""
    URL_PATH = "url_path"  # /api/v1/users vs /api/v2/users
    HEADER = "header"  # Accept-Version: 1.0
    QUERY_PARAM = "query_param"  # ?version=1.0
    CONTENT_TYPE = "content_type"  # application/vnd.api+json;version=1
    HYBRID = "hybrid"  # Support multiple strategies

class SemanticVersion:
    """Semantic versioning: MAJOR.MINOR.PATCH"""
    
    def __init__(self, major: int, minor: int, patch: int):
        self.major = major
        self.minor = minor
        self.patch = patch
    
    def __str__(self):
        return f"{self.major}.{self.minor}.{self.patch}"
    
    def __lt__(self, other):
        return (self.major, self.minor, self.patch) < (other.major, other.minor, other.patch)
    
    def __eq__(self, other):
        return (self.major, self.minor, self.patch) == (other.major, other.minor, other.patch)
    
    def is_breaking_change(self, other: 'SemanticVersion') -> bool:
        """Check if change is breaking (major version bump)"""
        return self.major > other.major

@dataclass
class APIVersion:
    """API version metadata"""
    version: SemanticVersion
    release_date: datetime
    deprecation_date: Optional[datetime] = None
    sunset_date: Optional[datetime] = None
    status: str = "active"  # active, deprecated, sunset
    breaking_changes: List[str] = None
    new_features: List[str] = None
    
    def __post_init__(self):
        if self.breaking_changes is None:
            self.breaking_changes = []
        if self.new_features is None:
            self.new_features = []
    
    def is_deprecated(self) -> bool:
        """Check if version is deprecated"""
        if self.deprecation_date is None:
            return False
        return datetime.now() >= self.deprecation_date
    
    def is_sunset(self) -> bool:
        """Check if version is sunset (no longer supported)"""
        if self.sunset_date is None:
            return False
        return datetime.now() >= self.sunset_date
    
    def days_until_deprecation(self) -> Optional[int]:
        """Days until version is deprecated"""
        if self.deprecation_date is None:
            return None
        delta = self.deprecation_date - datetime.now()
        return max(0, delta.days)
    
    def days_until_sunset(self) -> Optional[int]:
        """Days until version is sunset"""
        if self.sunset_date is None:
            return None
        delta = self.sunset_date - datetime.now()
        return max(0, delta.days)

class APIVersionRegistry:
    """Manage API versions"""
    
    def __init__(self):
        self.versions: Dict[str, APIVersion] = {}
        self.current_version: Optional[SemanticVersion] = None
        self.strategy: VersioningStrategy = VersioningStrategy.URL_PATH
    
    def register_version(self, api_version: APIVersion):
        """Register API version"""
        version_str = str(api_version.version)
        self.versions[version_str] = api_version
        
        if self.current_version is None or api_version.version > self.current_version:
            self.current_version = api_version.version
    
    def get_version(self, version_str: str) -> Optional[APIVersion]:
        """Get version by string"""
        return self.versions.get(version_str)
    
    def get_active_versions(self) -> List[APIVersion]:
        """Get all active versions"""
        return [v for v in self.versions.values() if v.status == "active"]
    
    def get_deprecated_versions(self) -> List[APIVersion]:
        """Get deprecated versions still supported"""
        return [v for v in self.versions.values() 
                if v.is_deprecated() and not v.is_sunset()]
    
    def get_sunset_versions(self) -> List[APIVersion]:
        """Get versions no longer supported"""
        return [v for v in self.versions.values() if v.is_sunset()]
    
    def deprecate_version(self, version_str: str, deprecation_date: datetime, sunset_date: datetime):
        """Mark version as deprecated"""
        version = self.get_version(version_str)
        if version:
            version.status = "deprecated"
            version.deprecation_date = deprecation_date
            version.sunset_date = sunset_date
    
    def get_recommended_upgrade_path(self, current_version_str: str) -> Optional[str]:
        """Suggest upgrade path for client"""
        current = self.get_version(current_version_str)
        
        if not current:
            return None
        
        if current.is_sunset():
            # Must upgrade to active version
            active = self.get_active_versions()
            if active:
                return str(active[0].version)
        
        if current.is_deprecated():
            # Suggest upgrade to current
            return str(self.current_version)
        
        return None

# Usage
registry = APIVersionRegistry()

v1 = APIVersion(
    version=SemanticVersion(1, 0, 0),
    release_date=datetime(2024, 1, 1),
    deprecation_date=datetime(2025, 1, 1),
    sunset_date=datetime(2025, 7, 1),
    new_features=["Basic user endpoint", "Auth"]
)

v2 = APIVersion(
    version=SemanticVersion(2, 0, 0),
    release_date=datetime(2025, 1, 1),
    breaking_changes=["Removed deprecated fields", "Changed response format"],
    new_features=["Pagination", "Filtering", "New relationships"]
)

registry.register_version(v1)
registry.register_version(v2)
```

### Deprecation Policy
```python
class DeprecationNotice:
    """Communicate deprecation to clients"""
    
    def __init__(self, feature: str, version_removed: str):
        self.feature = feature
        self.version_removed = version_removed
        self.deprecation_date: Optional[datetime] = None
        self.sunset_date: Optional[datetime] = None
        self.replacement: Optional[str] = None
        self.migration_guide: str = ""
    
    def set_timeline(self, deprecation_date: datetime, sunset_date: datetime):
        """Set deprecation timeline"""
        self.deprecation_date = deprecation_date
        self.sunset_date = sunset_date
    
    def get_deprecation_header(self) -> str:
        """HTTP header for deprecation notice"""
        notice = f'Deprecation={self.deprecation_date.isoformat()}'
        
        if self.sunset_date:
            notice += f'; Sunset={self.sunset_date.isoformat()}'
        
        if self.replacement:
            notice += f'; Link=<{self.replacement}>; rel="successor-version"'
        
        return notice

class DeprecationPolicy:
    """Enforce deprecation lifecycle"""
    
    def __init__(self, notice_period_months: int = 12, support_period_months: int = 18):
        self.notice_period = notice_period_months
        self.support_period = support_period_months
        self.deprecations: List[DeprecationNotice] = []
    
    def create_deprecation_notice(self, feature: str, version_removed: str) -> DeprecationNotice:
        """Create deprecation notice"""
        notice = DeprecationNotice(feature, version_removed)
        
        now = datetime.now()
        deprecation_date = now
        sunset_date = now + timedelta(days=self.support_period * 30)
        
        notice.set_timeline(deprecation_date, sunset_date)
        
        self.deprecations.append(notice)
        
        return notice
    
    def get_active_deprecations(self) -> List[DeprecationNotice]:
        """Get current deprecations"""
        now = datetime.now()
        
        return [d for d in self.deprecations 
                if d.deprecation_date <= now < d.sunset_date]
    
    def get_upcoming_sunsets(self, days: int = 90) -> List[DeprecationNotice]:
        """Get features approaching sunset"""
        now = datetime.now()
        cutoff = now + timedelta(days=days)
        
        return [d for d in self.deprecations 
                if now < d.sunset_date <= cutoff]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Breaking Changes & Migration

### Change Classification
```python
class ChangeType(Enum):
    """Types of API changes"""
    ADDITIVE = "additive"  # Safe - new fields/endpoints
    DEPRECATION = "deprecation"  # Safe - feature marked for removal
    MODIFICATION = "modification"  # Potentially breaking
    REMOVAL = "removal"  # Breaking - removed feature
    RENAME = "rename"  # Breaking - renamed field
    SIGNATURE_CHANGE = "signature_change"  # Breaking - response structure

class BreakingChange:
    """Document breaking change"""
    
    def __init__(self, feature: str, change_type: ChangeType, version: str):
        self.feature = feature
        self.change_type = change_type
        self.version = version
        self.description: str = ""
        self.migration_path: str = ""
        self.impact_level: str = "high"  # high, medium, low
    
    def set_migration_guidance(self, guidance: str):
        """Explain how clients should migrate"""
        self.migration_path = guidance

class MigrationGuide:
    """Help clients migrate between versions"""
    
    def __init__(self, from_version: str, to_version: str):
        self.from_version = from_version
        self.to_version = to_version
        self.changes: List[BreakingChange] = []
        self.code_examples: Dict[str, str] = {}  # feature -> example
    
    def add_change(self, change: BreakingChange):
        """Add breaking change"""
        self.changes.append(change)
    
    def add_code_example(self, feature: str, old_code: str, new_code: str):
        """Add before/after code example"""
        self.code_examples[feature] = f"""
## {feature}

**Before (v{self.from_version}):**
```
{old_code}
```

**After (v{self.to_version}):**
```
{new_code}
```
"""
    
    def generate_guide(self) -> str:
        """Generate migration guide"""
        guide = f"# Migration Guide: {self.from_version} → {self.to_version}\n\n"
        
        guide += "## Breaking Changes\n"
        for change in self.changes:
            guide += f"- **{change.feature}**: {change.description}\n"
        
        if self.code_examples:
            guide += "\n## Code Examples\n"
            for feature, example in self.code_examples.items():
                guide += example
        
        return guide

class ChangeValidator:
    """Validate changes between API versions"""
    
    @staticmethod
    def compare_schemas(old_schema: Dict, new_schema: Dict) -> List[BreakingChange]:
        """Detect breaking changes between schemas"""
        breaking_changes = []
        
        # Check for removed fields
        for field in old_schema:
            if field not in new_schema:
                breaking_changes.append(
                    BreakingChange(field, ChangeType.REMOVAL, "new")
                )
        
        # Check for type changes
        for field in old_schema:
            if field in new_schema:
                if old_schema[field] != new_schema[field]:
                    breaking_changes.append(
                        BreakingChange(field, ChangeType.SIGNATURE_CHANGE, "new")
                    )
        
        return breaking_changes
```

### Client Compatibility Testing
```python
class ClientCompatibilityTest:
    """Test client compatibility across versions"""
    
    def __init__(self, client_name: str, version: str):
        self.client_name = client_name
        self.version = version
        self.test_results: Dict[str, bool] = {}
        self.compatibility_matrix: Dict[str, Dict[str, bool]] = {}
    
    def test_endpoint(self, endpoint: str, method: str, expected_status: int) -> bool:
        """Test if endpoint works"""
        # Simplified - would make actual HTTP request
        key = f"{method} {endpoint}"
        is_working = True  # Assume success for example
        
        self.test_results[key] = is_working
        return is_working
    
    def build_compatibility_matrix(self, api_versions: List[str]) -> Dict[str, Dict[str, bool]]:
        """Test client against multiple API versions"""
        matrix = {}
        
        for api_version in api_versions:
            matrix[api_version] = {
                'compatible': True,  # Would test actual compatibility
                'warnings': [],
                'errors': []
            }
        
        self.compatibility_matrix = matrix
        return matrix
    
    def generate_report(self) -> Dict:
        """Generate compatibility report"""
        return {
            'client': self.client_name,
            'client_version': self.version,
            'api_compatibility': self.compatibility_matrix,
            'overall_status': 'compatible' if all(
                v['compatible'] for v in self.compatibility_matrix.values()
            ) else 'incompatible'
        }

class VersionUpgradeValidator:
    """Validate upgrade path for clients"""
    
    def __init__(self):
        self.test_suites: Dict[str, List[Callable]] = {}
    
    def register_upgrade_test(self, from_version: str, to_version: str, test_func: Callable):
        """Register upgrade test"""
        key = f"{from_version}->{to_version}"
        if key not in self.test_suites:
            self.test_suites[key] = []
        self.test_suites[key].append(test_func)
    
    def validate_upgrade(self, from_version: str, to_version: str) -> Dict:
        """Run upgrade validation tests"""
        key = f"{from_version}->{to_version}"
        tests = self.test_suites.get(key, [])
        
        results = []
        for test in tests:
            try:
                result = test()
                results.append({'passed': True, 'result': result})
            except Exception as e:
                results.append({'passed': False, 'error': str(e)})
        
        return {
            'upgrade_path': key,
            'tests_run': len(tests),
            'tests_passed': sum(1 for r in results if r['passed']),
            'upgrade_safe': all(r['passed'] for r in results),
            'details': results
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. API Gateway Versioning

### Version Routing
```python
class VersionRouter:
    """Route requests to correct API version"""
    
    def __init__(self, strategy: VersioningStrategy):
        self.strategy = strategy
        self.handlers: Dict[str, Callable] = {}
    
    def register_version_handler(self, version: str, handler: Callable):
        """Register handler for version"""
        self.handlers[version] = handler
    
    def extract_version(self, request: Dict) -> str:
        """Extract version from request"""
        if self.strategy == VersioningStrategy.URL_PATH:
            # e.g., /api/v1/users → "v1"
            path = request.get('path', '')
            parts = path.split('/')
            for part in parts:
                if part.startswith('v') and part[1:].split('.')[0].isdigit():
                    return part
        
        elif self.strategy == VersioningStrategy.HEADER:
            # e.g., Accept-Version: 1.0
            return request.get('headers', {}).get('Accept-Version', 'v1')
        
        elif self.strategy == VersioningStrategy.QUERY_PARAM:
            # e.g., ?version=1.0
            return request.get('query', {}).get('version', 'v1')
        
        return 'v1'  # Default
    
    def route_request(self, request: Dict) -> Dict:
        """Route request to appropriate version handler"""
        version = self.extract_version(request)
        
        handler = self.handlers.get(version)
        if not handler:
            return {'status': 404, 'error': f'Version {version} not found'}
        
        return handler(request)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. API Versioning Checklist

✅ **Versioning Strategy**
- [ ] Strategy selected (URL, header, etc.)
- [ ] Versioning scheme documented
- [ ] Routing logic implemented
- [ ] Default version defined
- [ ] Version detection working

✅ **Deprecation Management**
- [ ] Policy established
- [ ] Timeline defined
- [ ] Notices communicated
- [ ] Client alerts working
- [ ] Deprecation headers sent

✅ **Breaking Changes**
- [ ] Changes documented
- [ ] Migration guides written
- [ ] Code examples provided
- [ ] Tests created
- [ ] Support ready

✅ **Compatibility**
- [ ] Backwards compatibility tested
- [ ] Client versions tracked
- [ ] Upgrade paths validated
- [ ] Fallback routes available
- [ ] Error handling clear

✅ **Operations**
- [ ] Multiple versions running
- [ ] Load balanced correctly
- [ ] Monitoring per version
- [ ] Logging per version
- [ ] Metrics per version
</VSCode.Cell>
```